# 04 — Convergence & Limit Theorems
**References:** Wasserman (2004) Ch. 5 · Casella & Berger (2002) Ch. 5 · van der Vaart (1998) Ch. 2

## Narrative thread
```
Modes of convergence -> LLN -> CLT -> Slutsky -> CMT -> Delta method preview
```

## 1. Modes of convergence

Let $X_1, X_2, \ldots$ be a sequence of random variables. There are four main modes of convergence:

### Almost sure (a.s.) convergence
$$X_n \xrightarrow{a.s.} X \iff P\!\left(\lim_{n\to\infty} X_n = X\right) = 1$$

### Convergence in probability
$$X_n \xrightarrow{p} X \iff \forall\,\varepsilon > 0,\; P(|X_n - X| > \varepsilon) \to 0$$

### Convergence in distribution (weak convergence)
$$X_n \xrightarrow{d} X \iff F_{X_n}(x) \to F_X(x) \text{ at all continuity points of } F_X$$

### Convergence in $r$-th mean ($L^r$)
$$X_n \xrightarrow{L^r} X \iff E[|X_n - X|^r] \to 0$$

**Hierarchy of implications:**
$$X_n \xrightarrow{a.s.} X \;\Rightarrow\; X_n \xrightarrow{p} X \;\Rightarrow\; X_n \xrightarrow{d} X$$
$$X_n \xrightarrow{L^r} X \;\Rightarrow\; X_n \xrightarrow{p} X$$

> None of the reverse implications hold in general. Convergence in distribution is the weakest — it only constrains the marginal distributions, not the joint behavior of $(X_n, X)$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

plt.rcParams.update({
    'figure.dpi': 130, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': '#e8e8e8', 'axes.axisbelow': True,
    'axes.titlesize': 11, 'axes.titleweight': 'bold', 'legend.frameon': False,
})
np.random.seed(42)

# ── LLN: sample mean converges to population mean ─────────────────────────
mu_pop = 3.0
ns     = np.arange(1, 2001)
draws  = np.random.exponential(mu_pop, size=max(ns))

cumulative_means = np.cumsum(draws) / ns

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ax = axes[0]
for trial in range(8):
    d = np.random.exponential(mu_pop, size=max(ns))
    ax.plot(ns, np.cumsum(d)/ns, alpha=0.4, lw=1, color='#4361ee')
ax.axhline(mu_pop, color='#f72585', lw=2, linestyle='--', label=f'$\\mu={mu_pop}$')
ax.set_xscale('log')
ax.set_xlabel('n (log scale)'); ax.set_ylabel('$\\bar{X}_n$')
ax.set_title('Law of Large Numbers (SLLN)\n$\\bar{X}_n \\xrightarrow{a.s.} \\mu$ as $n\\to\\infty$')
ax.legend()

# ── Convergence in probability: P(|X_n - X| > eps) -> 0 ───────────────────
# X_n = Binomial(n, p)/n  -> p  (WLLN for Bernoulli)
p_true = 0.4
epsilon = 0.05
ns2     = np.array([10, 50, 100, 500, 1000, 5000, 10000])
probs_exceed = []
for n_mc in ns2:
    samples = np.random.binomial(n_mc, p_true, size=5000) / n_mc
    probs_exceed.append(np.mean(np.abs(samples - p_true) > epsilon))

axes[1].plot(ns2, probs_exceed, 'o-', color='#4361ee', lw=2, markersize=6)
axes[1].set_xscale('log')
axes[1].set_xlabel('n (log scale)')
axes[1].set_ylabel(f'$P(|\\bar{{X}}_n - p| > {epsilon})$')
axes[1].set_title(f'Convergence in probability\n$P(|\\bar{{X}}_n - {p_true}| > {epsilon}) \\to 0$')

plt.suptitle('Law of Large Numbers — Two Perspectives', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 2. Central Limit Theorem

**CLT (Lindeberg-Lévy):** Let $X_1, \ldots, X_n$ be i.i.d. with $E[X] = \mu$ and $\text{Var}(X) = \sigma^2 < \infty$. Then:

$$\sqrt{n}\,\frac{\bar{X}_n - \mu}{\sigma} \xrightarrow{d} \mathcal{N}(0,1)$$

Equivalently: $\bar{X}_n \xrightarrow{d} \mathcal{N}\!\left(\mu, \frac{\sigma^2}{n}\right)$

**Why it matters for inference:**
- Justifies using the normal distribution for large-sample confidence intervals and tests
- The rate of convergence is $O(1/\sqrt{n})$ — decreasing the SE by half requires $4\times$ the data

**Berry-Esseen bound:** The approximation error is bounded:
$$\sup_x \left|P\!\left(\sqrt{n}\,\frac{\bar{X}_n - \mu}{\sigma} \leq x\right) - \Phi(x)\right| \leq \frac{C\,E[|X-\mu|^3]}{\sigma^3 \sqrt{n}}$$

> **Rule of thumb:** $n \geq 30$ is sufficient for symmetric distributions; for highly skewed, $n \geq 100$ may be needed. Berry-Esseen gives the rigorous bound.

In [ ]:
# ── CLT across different parent distributions ──────────────────────────────
sample_sizes = [1, 5, 30, 100]
distributions = [
    ('Uniform(0,1)', lambda n: np.random.uniform(0, 1, n), 0.5, np.sqrt(1/12)),
    ('Exponential(1)', lambda n: np.random.exponential(1, n), 1.0, 1.0),
    ('Bernoulli(0.1)', lambda n: np.random.binomial(1, 0.1, n), 0.1, np.sqrt(0.1*0.9)),
]

fig, axes = plt.subplots(3, 4, figsize=(14, 10))
x_std = np.linspace(-4, 4, 300)

for row, (dist_name, sampler, mu_d, sigma_d) in enumerate(distributions):
    for col, n in enumerate(sample_sizes):
        n_reps = 8000
        means  = np.array([sampler(n).mean() for _ in range(n_reps)])
        z      = (means - mu_d) / (sigma_d / np.sqrt(n))

        ax = axes[row, col]
        ax.hist(z, bins=60, density=True, color='#4361ee', alpha=0.7, edgecolor='white', linewidth=0.2)
        ax.plot(x_std, stats.norm.pdf(x_std), 'r-', lw=2, label='N(0,1)')
        ax.set_xlim(-4, 4)
        ax.set_title(f'{dist_name}\nn={n}', fontsize=9)
        if row == 2:
            ax.set_xlabel('Standardized $\\bar{X}_n$')
        if col == 0:
            ax.set_ylabel('Density')

plt.suptitle('Central Limit Theorem — convergence to N(0,1) across parent distributions',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Slutsky's theorem and the Continuous Mapping Theorem

**Slutsky's theorem:** If $X_n \xrightarrow{d} X$ and $Y_n \xrightarrow{p} c$ (constant), then:
$$X_n + Y_n \xrightarrow{d} X + c \qquad X_n Y_n \xrightarrow{d} cX \qquad X_n/Y_n \xrightarrow{d} X/c \;(c \neq 0)$$

**Continuous Mapping Theorem (CMT):** If $g$ is continuous and $X_n \xrightarrow{d} X$, then $g(X_n) \xrightarrow{d} g(X)$.

**Practical use:** Replacing $\sigma$ by $\hat{\sigma}$ in the CLT:
$$\sqrt{n}\,\frac{\bar{X}_n - \mu}{\hat{\sigma}_n} \xrightarrow{d} \mathcal{N}(0,1)$$
This follows by Slutsky since $\hat{\sigma}_n \xrightarrow{p} \sigma$.

## 4. Delta method

The **delta method** extends the CLT to smooth functions of estimators.

If $\sqrt{n}(\hat{\theta}_n - \theta) \xrightarrow{d} \mathcal{N}(0, \sigma^2)$ and $g$ is differentiable at $\theta$ with $g'(\theta) \neq 0$:

$$\sqrt{n}(g(\hat{\theta}_n) - g(\theta)) \xrightarrow{d} \mathcal{N}(0,\, [g'(\theta)]^2 \sigma^2)$$

**Multivariate delta method:** If $\sqrt{n}(\hat{\boldsymbol{\theta}}_n - \boldsymbol{\theta}) \xrightarrow{d} \mathcal{N}(\mathbf{0}, \boldsymbol{\Sigma})$:
$$\sqrt{n}(g(\hat{\boldsymbol{\theta}}_n) - g(\boldsymbol{\theta})) \xrightarrow{d} \mathcal{N}(0,\, \nabla g(\boldsymbol{\theta})^\top \boldsymbol{\Sigma}\, \nabla g(\boldsymbol{\theta}))$$

In [ ]:
# ── Delta method: SE of log(p_hat) for Bernoulli(p) ──────────────────────
# p_hat = X_bar ~ N(p, p(1-p)/n)  by CLT
# g(p) = log(p),  g'(p) = 1/p
# => sqrt(n)(log(p_hat) - log(p)) -> N(0, (1-p)/p)  by delta method

p_true = 0.3
n_obs  = 200
n_reps = 20_000

p_hats  = np.random.binomial(n_obs, p_true, n_reps) / n_obs
p_hats  = p_hats[p_hats > 0]  # avoid log(0)

# Standardized log(p_hat)
delta_se  = np.sqrt((1 - p_true) / p_true)  # delta method SE * sqrt(n)
z_delta   = np.sqrt(n_obs) * (np.log(p_hats) - np.log(p_true)) / delta_se

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

x_std = np.linspace(-4, 4, 300)

axes[0].hist(z_delta, bins=80, density=True, color='#4361ee', alpha=0.7, edgecolor='white')
axes[0].plot(x_std, stats.norm.pdf(x_std), 'r-', lw=2, label='N(0,1)')
axes[0].set_title(f'Delta method: $\\sqrt{{n}}(\\log\\hat{{p}} - \\log p) / \\sqrt{{(1-p)/p}}$\n'
                   f'$p={p_true}$, $n={n_obs}$ — should be N(0,1)')
axes[0].legend()

# QQ plot
z_sorted = np.sort(z_delta)
theo_q   = stats.norm.ppf(np.linspace(0.001, 0.999, len(z_sorted)))
axes[1].scatter(theo_q, z_sorted, s=2, alpha=0.3, color='#4361ee')
axes[1].plot([-4, 4], [-4, 4], 'r-', lw=2)
axes[1].set_xlabel('Theoretical N(0,1) quantiles')
axes[1].set_ylabel('Sample quantiles of $\\sqrt{n}(\\log\\hat{p} - \\log p)/SE$')
axes[1].set_title('Q-Q plot: Delta method approximation quality')

plt.suptitle('Delta Method Verification — Binomial log-proportion', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()
print(f'Delta method SE for log(p): {delta_se/np.sqrt(n_obs):.4f}')
print(f'Empirical SE of log(p_hat): {np.log(p_hats).std():.4f}')

## 5. Key takeaways

| Theorem | Statement | Used for |
|---|---|---|
| SLLN | $\bar{X}_n \xrightarrow{a.s.} \mu$ | Consistency of estimators |
| CLT | $\sqrt{n}(\bar{X}_n-\mu)/\sigma \xrightarrow{d} N(0,1)$ | Large-sample CIs and tests |
| Slutsky | Replace $\sigma$ by $\hat{\sigma}$ in CLT | Feasible inference |
| CMT | $g(X_n) \xrightarrow{d} g(X)$ | Nonlinear functions of estimates |
| Delta method | SE of $g(\hat{\theta})$ via $g'(\theta)\cdot\text{SE}(\hat{\theta})$ | Inference on transformations |

**Next:** notebook 05 — point estimation theory, where these tools are used to derive and evaluate estimators.